In [1]:
from flask import Flask, request, jsonify, render_template_string
from werkzeug.utils import secure_filename
import tensorflow as tf
from tensorflow import keras
import numpy as np
from PIL import Image
import os
import base64
from io import BytesIO
import threading

In [2]:
PROJECT_DIR = r"D:\CSE Code\CSE465 Project"
MODEL_PATH = os.path.join(PROJECT_DIR, "model", "banknote_classifier.h5")
UPLOAD_FOLDER = os.path.join(PROJECT_DIR, "uploads")
os.makedirs(UPLOAD_FOLDER, exist_ok=True)

IMG_SIZE = 224
ALLOWED_EXTENSIONS = {'png', 'jpg', 'jpeg'}

# Cell 3: Load Model
print("Loading model...")
model = keras.models.load_model(MODEL_PATH)
print("Model loaded successfully!")

Loading model...


Model loaded successfully!


In [3]:
def allowed_file(filename):
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in ALLOWED_EXTENSIONS

def preprocess_image(image_path):
    img = Image.open(image_path)
    img = img.convert('RGB')
    img = img.resize((IMG_SIZE, IMG_SIZE))
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    return img_array

def predict_banknote(image_path):
    img_array = preprocess_image(image_path)
    prediction = model.predict(img_array, verbose=0)[0][0]
    
    if prediction < 0.5:
        result = "FAKE"
        confidence = (1 - prediction) * 100
    else:
        result = "REAL"
        confidence = prediction * 100
    
    return result, confidence


In [4]:
HTML_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <title>NoteShieldBD - Banknote Authentication</title>
    <style>
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }
        body {
            font-family: 'Segoe UI', Arial, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            display: flex;
            justify-content: center;
            align-items: center;
            padding: 20px;
        }
        .container {
            background: white;
            border-radius: 20px;
            box-shadow: 0 20px 60px rgba(0,0,0,0.3);
            max-width: 600px;
            width: 100%;
            padding: 40px;
            text-align: center;
        }
        h1 {
            color: #333;
            margin-bottom: 10px;
            font-size: 32px;
        }
        .subtitle {
            color: #666;
            margin-bottom: 30px;
        }
        .shield-icon {
            font-size: 80px;
            margin-bottom: 20px;
        }
        .upload-area {
            border: 3px dashed #667eea;
            border-radius: 15px;
            padding: 40px 20px;
            margin: 30px 0;
            cursor: pointer;
            transition: all 0.3s;
            background: #f8f9ff;
        }
        .upload-area:hover {
            border-color: #764ba2;
            background: #f0f2ff;
        }
        input[type="file"] {
            display: none;
        }
        .btn {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            border: none;
            padding: 15px 40px;
            font-size: 18px;
            border-radius: 50px;
            cursor: pointer;
            transition: all 0.3s;
            margin-top: 20px;
        }
        .btn:hover {
            transform: translateY(-2px);
            box-shadow: 0 7px 20px rgba(102,126,234,0.6);
        }
        .result {
            margin-top: 30px;
            padding: 20px;
            border-radius: 15px;
            display: none;
        }
        .result.real {
            background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%);
            color: white;
        }
        .result.fake {
            background: linear-gradient(135deg, #ee0979 0%, #ff6a00 100%);
            color: white;
        }
        .result h2 {
            font-size: 36px;
            margin-bottom: 10px;
        }
        .preview {
            max-width: 100%;
            max-height: 300px;
            border-radius: 10px;
            margin: 20px 0;
            display: none;
        }
        .loading {
            display: none;
            margin: 20px 0;
        }
        .spinner {
            border: 4px solid #f3f3f3;
            border-top: 4px solid #667eea;
            border-radius: 50%;
            width: 40px;
            height: 40px;
            animation: spin 1s linear infinite;
            margin: 0 auto;
        }
        @keyframes spin {
            0% { transform: rotate(0deg); }
            100% { transform: rotate(360deg); }
        }
    </style>
</head>
<body>
    <div class="container">
        <div class="shield-icon">🛡️</div>
        <h1>NoteShieldBD</h1>
        <p class="subtitle">Bangladeshi Banknote Authentication System</p>
        
        <form id="upload-form" enctype="multipart/form-data">
            <div class="upload-area" onclick="document.getElementById('file-input').click()">
                <div style="font-size: 60px; margin-bottom: 15px;">📤</div>
                <div style="font-size: 18px; margin-bottom: 10px;">Click to upload or drag and drop</div>
                <div style="font-size: 14px; color: #999;">PNG, JPG or JPEG</div>
            </div>
            <input type="file" id="file-input" name="file" accept="image/*" required>
            <img id="preview" class="preview" src="" alt="Preview">
            <button type="submit" class="btn">Authenticate Banknote</button>
        </form>
        
        <div class="loading" id="loading">
            <div class="spinner"></div>
            <p style="margin-top: 10px;">Analyzing...</p>
        </div>
        
        <div class="result" id="result">
            <h2 id="result-text"></h2>
            <p id="confidence-text" style="font-size: 24px;"></p>
        </div>
    </div>
    
    <script>
        const fileInput = document.getElementById('file-input');
        const preview = document.getElementById('preview');
        const form = document.getElementById('upload-form');
        const loading = document.getElementById('loading');
        const result = document.getElementById('result');
        
        fileInput.addEventListener('change', function() {
            const file = this.files[0];
            if (file) {
                const reader = new FileReader();
                reader.onload = function(e) {
                    preview.src = e.target.result;
                    preview.style.display = 'block';
                    result.style.display = 'none';
                };
                reader.readAsDataURL(file);
            }
        });
        
        form.addEventListener('submit', async function(e) {
            e.preventDefault();
            
            const formData = new FormData();
            formData.append('file', fileInput.files[0]);
            
            loading.style.display = 'block';
            result.style.display = 'none';
            
            try {
                const response = await fetch('/predict', {
                    method: 'POST',
                    body: formData
                });
                
                const data = await response.json();
                
                loading.style.display = 'none';
                result.style.display = 'block';
                result.className = 'result ' + data.result.toLowerCase();
                
                document.getElementById('result-text').textContent = data.result;
                document.getElementById('confidence-text').textContent = 
                    'Confidence: ' + data.confidence.toFixed(2) + '%';
            } catch (error) {
                loading.style.display = 'none';
                alert('Error: ' + error.message);
            }
        });
    </script>
</body>
</html>
"""


In [5]:
app = Flask(__name__)
app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER
app.config['MAX_CONTENT_LENGTH'] = 16 * 1024 * 1024

@app.route('/')
def index():
    return render_template_string(HTML_TEMPLATE)

@app.route('/predict', methods=['POST'])
def predict():
    if 'file' not in request.files:
        return jsonify({'error': 'No file uploaded'}), 400
    
    file = request.files['file']
    
    if file.filename == '':
        return jsonify({'error': 'No file selected'}), 400
    
    if file and allowed_file(file.filename):
        filename = secure_filename(file.filename)
        filepath = os.path.join(app.config['UPLOAD_FOLDER'], filename)
        file.save(filepath)
        
        result, confidence = predict_banknote(filepath)
        
        return jsonify({
            'result': result,
            'confidence': float(confidence),
            'filename': filename
        })
    else:
        return jsonify({'error': 'Invalid file type'}), 400

print("Flask app created successfully!")


Flask app created successfully!


In [6]:
def run_app():
    app.run(host='127.0.0.1', port=5000, debug=False, use_reloader=False)

# Start Flask in a separate thread
thread = threading.Thread(target=run_app, daemon=True)
thread.start()

print("\n" + "="*60)
print("🚀 Flask Web Application is Running!")
print("="*60)
print("\n📍 Open your browser and go to: http://127.0.0.1:5000/")
print("\n⚠️ Note: To stop the server, restart the Jupyter kernel")
print("="*60)


🚀 Flask Web Application is Running!

📍 Open your browser and go to: http://127.0.0.1:5000/

⚠️ Note: To stop the server, restart the Jupyter kernel
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [15/Dec/2025 16:42:20] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Dec/2025 16:42:21] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [15/Dec/2025 16:42:38] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Dec/2025 16:43:10] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Dec/2025 16:43:20] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/Dec/2025 16:43:30] "POST /predict HTTP/1.1" 200 -
